# Immune Data Pretraining


In [60]:
from collator import ScImmuneDataCollator
from datasets import load_from_disk
import torch
from tokenizer import ScImmuneTokenizer
from torch.utils.data import DataLoader


In [67]:
tokenizer = ScImmuneTokenizer(vocab_file="vocab_with_metadata.json")

In [ ]:
## Set data folder
tokenized_data_path = 'scimmune-model/tokenized_data'

## Load dataset
tokenized_dataset = load_from_disk(tokenized_data_path)

### Post-processing tokenized output

In [34]:
# Number of metadata tokens you want to add zeros for
num_metadata_tokens = 6


def prepend_zeros(example):
    zeros = torch.zeros(num_metadata_tokens, dtype=example["values"].dtype)
    example["values"] = torch.cat((zeros, example["values"]))
    return example

# Map over dataset
tokenized_dataset = tokenized_dataset.map(prepend_zeros)

In [35]:
cls_token_id = 60695  # ID for <cls>

def move_cls_to_front(example):
    genes = example["genes"].tolist()   # convert tensor → list

    if cls_token_id in genes:
        idx = genes.index(cls_token_id)

        # Move CLS to the front
        genes.pop(idx)

        genes.insert(0, cls_token_id)

    # Convert back to tensors for HF dataset
    example["genes"] = torch.tensor(genes, dtype=torch.long)

    return example

tokenized_dataset = tokenized_dataset.map(move_cls_to_front)

In [36]:
tokenized_dataset = tokenized_dataset.rename_column("values", "expressions")
tokenized_dataset[0]

{'genes': tensor([60695, 60801, 60926, 60942, 61043, 61046, 61045,  4441,  6019,  8023,
          3933,  2114,  3573,  4452, 15664,  3341, 16375, 30442, 11130, 60697,
         20091,  9801, 20797, 16339, 10113, 18757, 32057, 21084,  4029,  5216,
         34889, 32342,  5254, 20881, 21348, 30333,  4453,  8712,  7702,  1448,
         20880,  3708, 11047, 19063, 18166, 19580, 19850, 16237,  8200, 16406,
         33798,  8149,  8799,  9567, 35766, 11015,  3985, 12916,  7867,  8356,
         18305,  9672, 35798, 60697, 20316, 34072, 18889, 35668, 31856, 10632,
         17603, 19165, 36514, 19205,  4378,  4379, 12828, 19593, 16871, 35305,
          5242,  1519, 17211,  3834,  5106,  8732, 31534,  8376, 35021, 32258,
         21424, 12146, 18900,  4917, 60697, 20516,  1878, 30350, 34488,  2936,
          2581, 33868,  3452,  7854, 10948, 33996, 18858, 30316,  8334, 33878,
         30431, 31983, 21071, 20109, 32368, 34605, 21440, 32588, 30686, 35739,
         12392, 16647, 12352, 30308, 30370,

## Collator

In [62]:
ds = tokenized_dataset.with_format(type="torch", columns=["genes","expressions"])
num_metadata_tokens = 6

In [59]:
pad_token_id_value = 60694

collator = ScImmuneDataCollator(
    do_padding=True,
    pad_token_id=pad_token_id_value,  # match your vocab
    pad_value=0,
    do_mlm=True,
    do_binning=True,       # applies  preprocess.binning (51 bins in your code)
    mlm_probability=0.15,
    mask_value=-1,
    max_length=1200,
    sampling=True,
    keep_first_n_tokens=1 + num_metadata_tokens,  # <cls> + metadata are preserved
    data_style="pcpt",     # "pcpt" or "both" are typical for pretraining
)

In [63]:
dl = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collator)
batch = next(iter(dl))

print({k: (v.shape, v.dtype) for k,v in batch.items()})

{'gene': (torch.Size([8, 161]), torch.int64), 'expr': (torch.Size([8, 161]), torch.float32), 'masked_expr': (torch.Size([8, 161]), torch.float32)}


### Collator Testing

In [64]:
pad_id = pad_token_id_value
keep = 1 + num_metadata_tokens
n_bins = 51

In [66]:
# 4.1 shapes match
assert batch["gene"].shape == batch["expr"].shape == batch["masked_expr"].shape

# 4.2 padding behavior
assert (batch["expr"][batch["gene"] == pad_id] == 0).all()        # expr pad_value
assert (batch["gene"] == pad_id).any() or True                     # padding exists or not

# 4.3 prefix untouched (no masking/binning)
# masked_expr must equal expr in prefix; prefix should be zeros if you set them that way
assert torch.equal(batch["masked_expr"][:, :keep], batch["expr"][:, :keep])

# 4.4 post-prefix binned range
post = batch["expr"][:, keep:]
assert torch.isfinite(post).all()
assert (post >= 0).all() & (post < n_bins).all()
assert torch.allclose(post, post.round())  # integer-like after binning

In [ ]:
row = 0
prefix_ids = batch["gene"][row, :keep].tolist()
prefix_tokens = tokenizer.convert_ids_to_tokens(prefix_ids)
print("Prefix tokens:", prefix_tokens)  # expect: ['<cls>', '<cell_type=...>', '<tissue=...>', ...]

print("Prefix expr :", batch["expr"][row, :keep].tolist())         # expect zeros (or unbinned if you chose)
print("Prefix masked:", batch["masked_expr"][row, :keep].tolist()) # should match expr (no -1 here)

Prefix tokens: ['<cls>', '<cell_type=CL:0000905>', '<self_reported_ethnicity=HANCESTRO:0005>', '<tissue_general=UBERON:0000178>', '<development_stage=HsapDv:0000266>', '<sex=PATO:0000383>', '<disease=DOID:4>']
Prefix expr : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Prefix masked: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


: 

In [45]:
# from model import ScImmuneModel
# from config import ScImmuneConfig

# config = ScImmuneConfig.from_pretrained("scimmune-model") # load config locally

# model = ScImmuneModel(config)
# model